# Voice-cloning TTS test (F5-TTS vs Chatterbox), v2

Personal test, not part of the cyv repo.

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Runtime -> **Run all**.
3. Cell 2 asks you to upload **`ref.wav`** (from `~/Downloads/tts-clone-test/`).
4. The last cell downloads **`tts_out.zip`**.

This version **stops with a red error** if any step fails.
If that happens, copy the last 30 lines of that cell's output.


In [ ]:
# 1. Helper + GPU check. Every command below stops the notebook if it fails.
import subprocess, sys, os, glob

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-6000:])
    if p.returncode != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')


In [ ]:
# 2. Upload ref.wav and set the test sentences
import json
from google.colab import files
os.makedirs('/content/test/out', exist_ok=True)
os.chdir('/content/test')
up = files.upload()
assert os.path.exists('ref.wav'), 'please upload a file named ref.wav'
print('ref.wav bytes:', os.path.getsize('ref.wav'))

SENTENCES = [
    "Hello, this is a test of my cloned voice. How does this sound to you?",
    "I was thinking we could grab lunch tomorrow, maybe around one, if that works for you.",
    "Honestly, I did not expect the results to be this good. Let me know what you think.",
]
json.dump(SENTENCES, open('sentences.json', 'w'))


In [ ]:
# 3. Install uv
sh('pip install uv')


In [ ]:
# 4. F5-TTS: install into its own venv (a few minutes)
sh('uv venv /content/venv_f5 --python 3.11')
sh('VIRTUAL_ENV=/content/venv_f5 uv pip install f5-tts')
sh('/content/venv_f5/bin/python -c "import f5_tts, torch; print(\'f5 ok, cuda:\', torch.cuda.is_available())"')


In [ ]:
%%writefile /content/test/run_f5.py
# F5-TTS generation (ref_text empty -> it transcribes ref.wav itself)
import json, time
from f5_tts.api import F5TTS
sents = json.load(open('sentences.json'))
tts = F5TTS()
for i, s in enumerate(sents, 1):
    t = time.time()
    tts.infer(ref_file='ref.wav', ref_text='', gen_text=s,
              file_wave=f'out/f5_s{i}.wav', seed=42)
    print(f'f5 s{i}: {time.time()-t:.1f}s', flush=True)


In [ ]:
# 5. F5-TTS: generate
sh('cd /content/test && /content/venv_f5/bin/python run_f5.py')
made = sorted(glob.glob('/content/test/out/f5_s*.wav'))
print(made)
assert len(made) == 3, 'F5 did not write its 3 files'


In [ ]:
# 6. Chatterbox: install into its own venv
sh('uv venv /content/venv_cb --python 3.11')
sh('VIRTUAL_ENV=/content/venv_cb uv pip install chatterbox-tts')
sh('/content/venv_cb/bin/python -c "import chatterbox, torch; print(\'cb ok, cuda:\', torch.cuda.is_available())"')


In [ ]:
%%writefile /content/test/run_cb.py
# Chatterbox generation (defaults: exaggeration 0.5, cfg 0.5)
import json, time
import torchaudio as ta
from chatterbox.tts import ChatterboxTTS
sents = json.load(open('sentences.json'))
model = ChatterboxTTS.from_pretrained(device='cuda')
for i, s in enumerate(sents, 1):
    t = time.time()
    wav = model.generate(s, audio_prompt_path='ref.wav')
    ta.save(f'out/cb_s{i}.wav', wav, model.sr)
    print(f'cb s{i}: {time.time()-t:.1f}s', flush=True)


In [ ]:
# 7. Chatterbox: generate
sh('cd /content/test && /content/venv_cb/bin/python run_cb.py')
made = sorted(glob.glob('/content/test/out/cb_s*.wav'))
print(made)
assert len(made) == 3, 'Chatterbox did not write its 3 files'


In [ ]:
# 8. Quick listen in Colab (optional)
from IPython.display import Audio, display
for f in sorted(glob.glob('/content/test/out/*.wav')):
    print(f); display(Audio(f))


In [ ]:
# 9. Zip and download (refuses to zip an empty folder)
wavs = sorted(glob.glob('/content/test/out/*.wav'))
print(len(wavs), 'files:', [os.path.basename(w) for w in wavs])
assert wavs, 'out/ is empty: scroll up to the first red error'
sh('cd /content/test && rm -f tts_out.zip && zip -r tts_out.zip out')
files.download('/content/test/tts_out.zip')
